# Adrianic v4.1 — Corrected Mixed-Corpus Order / Interleaving Transfer

This fixes the v4.0 combined-corpus construction and explicitly tests:

- Shakespeare only,
- Bible only,
- Shakespeare → Bible,
- Bible → Shakespeare,
- balanced Shakespeare/Bible interleaving.

It retains the deterministic HOLD / evidence / selective replay / parity framework, and adds held-out rephrasing questions.

No semantic labels are used for model training or memory selection.

In [ ]:

from __future__ import annotations
from pathlib import Path
from collections import Counter
import argparse, hashlib, json, math, random, re, urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ---------------------------------------------------------------------
# Portable root
# ---------------------------------------------------------------------
if Path("/content").exists():
    ROOT = Path("/content")
elif Path("/mnt/data").exists():
    ROOT = Path("/mnt/data")
else:
    ROOT = Path.cwd()

OUT = ROOT / "adrianic_v41_order_interleaving_results"
OUT.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[a-zA-Z]+|[0-9]+|[^\w\s]")
STOP = set("the a an is are in of to and this that with for near under before after through beyond".split())

def seed_all(seed:int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def tokenize(text:str):
    return TOKEN_RE.findall(text.lower())

# ---------------------------------------------------------------------
# Controlled reality/STEM relationships.
# Labels below are EVALUATION ONLY.
# ---------------------------------------------------------------------
FACTS = [
    ("two plus two equals","four","math"),
    ("three plus five equals","eight","math"),
    ("seven minus three equals","four","math"),
    ("six times seven equals","fortytwo","math"),
    ("the square root of nine is","three","math"),
    ("a triangle has","three","math"),
    ("a square has","four","math"),
    ("an angle measured at ninety degrees is","right","math"),
    ("the ratio of circumference to diameter is called","pi","math"),
    ("the additive identity is","zero","math"),

    ("the si unit of force is the","newton","physics"),
    ("the si unit of energy is the","joule","physics"),
    ("the si unit of power is the","watt","physics"),
    ("the si unit of electric current is the","ampere","physics"),
    ("mass times acceleration gives","force","physics"),
    ("energy associated with motion is","kinetic","physics"),
    ("energy stored by position can be","potential","physics"),
    ("the force attracting masses is","gravity","physics"),
    ("electric charge is measured in","coulombs","physics"),
    ("light in vacuum travels extremely fast per","second","physics"),

    ("the chemical symbol for hydrogen is","h","chemistry"),
    ("the chemical symbol for oxygen is","o","chemistry"),
    ("the chemical symbol for carbon is","c","chemistry"),
    ("water contains hydrogen and","oxygen","chemistry"),
    ("common table salt contains sodium and","chlorine","chemistry"),
    ("an atom with six protons is","carbon","chemistry"),
    ("an atom with eight protons is","oxygen","chemistry"),
    ("a substance with ph below seven is","acidic","chemistry"),
    ("a substance with ph above seven is","basic","chemistry"),
    ("the center of an atom is the","nucleus","chemistry"),

    ("the planet closest to the sun is","mercury","world"),
    ("earth orbits the","sun","world"),
    ("the moon orbits the","earth","world"),
    ("the largest planet in the solar system is","jupiter","world"),
    ("liquid water freezes near zero degrees","celsius","world"),
    ("liquid water boils near one hundred degrees","celsius","world"),
    ("plants use light in","photosynthesis","world"),
    ("humans breathe in","oxygen","world"),
    ("the gas most abundant in earth atmosphere is","nitrogen","world"),
    ("solid liquid and gas are states of","matter","world"),
]

# Held-out phrasings are never used to construct the STEM training stream.
HELDOUT_QUESTIONS = [
    ("in physics mass times acceleration is","force"),
    ("the unit used for force is the","newton"),
    ("the unit used for energy is the","joule"),
    ("water contains hydrogen plus","oxygen"),
    ("six protons identifies","carbon"),
    ("eight protons identifies","oxygen"),
    ("earth travels around the","sun"),
    ("the moon travels around the","earth"),
    ("the biggest planet in the solar system is","jupiter"),
    ("plants using light is","photosynthesis"),
    ("earth atmosphere mostly contains","nitrogen"),
    ("solid liquid and gas describe","matter"),
]

TRAIN_TEMPLATES = [
    "{p} {a}.",
    "fact: {p} {a}.",
    "remember that {p} {a}.",
    "in this lesson {p} {a}.",
]

def _noise_code(n:int):
    # alphabetic unique token so junk is one-off without special labels
    alphabet="abcdefghijklmnopqrstuvwxyz"
    out=""
    x=n+1
    while x:
        x,r=divmod(x-1,26)
        out=alphabet[r]+out
    return "noise"+out

def make_stem_corpus(seed=0,repeats=5,junk=120):
    rng=random.Random(seed)
    sents=[]; labels=[]
    for p,a,d in FACTS:
        for r in range(repeats):
            sents.append(TRAIN_TEMPLATES[r % len(TRAIN_TEMPLATES)].format(p=p,a=a))
            labels.append(("fact",d,p,a))
    cursor=0
    for _ in range(junk):
        words=[_noise_code(cursor+k) for k in range(7)]
        cursor += 7
        rng.shuffle(words)
        sents.append(" ".join(words)+".")
        labels.append(("junk","junk","",""))
    order=list(range(len(sents)))
    rng.shuffle(order)
    return [sents[i] for i in order],[labels[i] for i in order]

# ---------------------------------------------------------------------
# Proxy corpora for local verification.
# These are deliberately different streams so order/interleaving logic
# can be tested without internet access.
# ---------------------------------------------------------------------
def proxy_A(seed=0,n=260):
    rng=random.Random(seed)
    names=["arden","cassian","lyra","orion","mara","selene","tavian","viola"]
    verbs=["whispers","wanders","argues","laughs","waits","returns","dreams","speaks"]
    objects=["beside the gate","through the hall","near the river","under the tower",
             "after the feast","beyond the garden","before the crowd","under the moon"]
    return [f"{rng.choice(names)} {rng.choice(verbs)} {rng.choice(objects)}." for _ in range(n)]

def proxy_B(seed=0,n=260):
    rng=random.Random(seed+7000)
    openings=["and behold","therefore","then came","it was written","hear now","thus spoke"]
    actors=["the traveler","the elder","the keeper","the child","the witness","the stranger"]
    actions=["crossed the valley","entered the city","remembered the covenant","counted the days",
             "returned at dawn","stood by the water","spoke to the people","watched the stars"]
    return [f"{rng.choice(openings)}, {rng.choice(actors)} {rng.choice(actions)}." for _ in range(n)]

# ---------------------------------------------------------------------
# Natural corpora
# ---------------------------------------------------------------------
def download_public_corpora():
    urls={
        "shakespeare":"https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "bible":"https://www.gutenberg.org/cache/epub/10900/pg10900.txt",
    }
    paths={}
    for name,url in urls.items():
        p=ROOT/f"{name}_corpus.txt"
        if not p.exists():
            print("downloading",name)
            urllib.request.urlretrieve(url,p)
        paths[name]=p
    return paths

def clean_sentences(text,min_tokens=4,max_tokens=80):
    raw=re.split(r"(?<=[.!?])\s+|\n{2,}",text)
    return [s.strip().replace("\n"," ") for s in raw
            if min_tokens <= len(tokenize(s)) <= max_tokens]

def deterministic_sample(xs,n,seed):
    xs=list(xs)
    rng=random.Random(seed)
    if len(xs)<=n:
        rng.shuffle(xs); return xs
    idx=rng.sample(range(len(xs)),n)
    return [xs[i] for i in idx]

def interleave_equal(a,b,seed):
    # Exactly balanced, then locally shuffle each A/B pair orientation.
    n=min(len(a),len(b))
    a=list(a[:n]); b=list(b[:n])
    rng=random.Random(seed)
    out=[]
    for x,y in zip(a,b):
        if rng.random()<.5: out.extend([x,y])
        else: out.extend([y,x])
    return out

def corpus_conditions(seed=0,full=False,per_source=220):
    if full:
        paths=download_public_corpora()
        A_all=clean_sentences(paths["shakespeare"].read_text(errors="ignore"))
        B_all=clean_sentences(paths["bible"].read_text(errors="ignore"))
        A_name="shakespeare"; B_name="bible"
    else:
        A_all=proxy_A(seed,per_source*2)
        B_all=proxy_B(seed,per_source*2)
        A_name="proxy_A"; B_name="proxy_B"

    A=deterministic_sample(A_all,per_source,seed+11)
    B=deterministic_sample(B_all,per_source,seed+29)

    # THE BUG FIX:
    # mixed conditions are constructed from independently sampled A and B lists,
    # never by concatenating raw text and truncating the front.
    conditions={
        f"{A_name}_only": list(A),
        f"{B_name}_only": list(B),
        f"{A_name}_then_{B_name}": list(A)+list(B),
        f"{B_name}_then_{A_name}": list(B)+list(A),
        "balanced_interleaved": interleave_equal(A,B,seed+101),
    }
    provenance={
        f"{A_name}_only": (len(A),0),
        f"{B_name}_only": (0,len(B)),
        f"{A_name}_then_{B_name}": (len(A),len(B)),
        f"{B_name}_then_{A_name}": (len(A),len(B)),
        "balanced_interleaved": (len(A),len(B)),
    }
    return conditions,provenance,A_name,B_name

# ---------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------
class Vocab:
    def __init__(self,texts,max_vocab=8000):
        c=Counter()
        for t in texts:c.update(tokenize(t))
        self.itos=["<pad>","<unk>"]+[w for w,_ in c.most_common(max_vocab-2)]
        self.stoi={w:i for i,w in enumerate(self.itos)}
    def encode(self,text):
        return [self.stoi.get(w,1) for w in tokenize(text)]

class TinySSM(nn.Module):
    def __init__(self,vocab_size,emb=40,hid=64):
        super().__init__()
        self.emb=nn.Embedding(vocab_size,emb)
        self.rnn=nn.RNN(emb,hid,nonlinearity="tanh",batch_first=True)
        self.out=nn.Linear(hid,vocab_size)
    def forward(self,x,h=None):
        e=self.emb(x)
        z,h=self.rnn(e,h)
        return self.out(z),h

def _model_device(model):
    return next(model.parameters()).device

def train_sentence(model,opt,ids):
    if len(ids)<2:return float("nan")
    device=_model_device(model)
    x=torch.tensor(ids[:-1],dtype=torch.long,device=device)[None,:]
    y=torch.tensor(ids[1:],dtype=torch.long,device=device)[None,:]
    opt.zero_grad(set_to_none=True)
    logits,_=model(x)
    loss=nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    opt.step()
    return float(loss.item())

def seq_loss(model,ids):
    if len(ids)<2:return float("nan")
    device=_model_device(model)
    x=torch.tensor(ids[:-1],dtype=torch.long,device=device)[None,:]
    y=torch.tensor(ids[1:],dtype=torch.long,device=device)[None,:]
    with torch.no_grad():
        logits,_=model(x)
        return float(nn.functional.cross_entropy(
            logits.reshape(-1,logits.size(-1)),y.reshape(-1)).item())

def answer_rank(model,vocab,prompt,answer):
    ids=vocab.encode(prompt)
    if not ids:return 10**9
    device=_model_device(model)
    x=torch.tensor(ids,dtype=torch.long,device=device)[None,:]
    with torch.no_grad():
        logits,_=model(x)
    v=logits[0,-1]
    aid=vocab.stoi.get(answer,1)
    return int((v>v[aid]).sum().item())+1

def qa_metrics(model,vocab,heldout=False):
    pairs=HELDOUT_QUESTIONS if heldout else [(p,a) for p,a,_ in FACTS]
    ranks=[answer_rank(model,vocab,p,a) for p,a in pairs]
    return {
        "top1":float(np.mean([r==1 for r in ranks])),
        "top5":float(np.mean([r<=5 for r in ranks])),
        "mrr":float(np.mean([1.0/r for r in ranks])),
        "median_rank":float(np.median(ranks)),
    }

# ---------------------------------------------------------------------
# Deterministic evidence graph / memory
# ---------------------------------------------------------------------
def chunk_features(sentences,vocab,model):
    counts=Counter()
    chunks=[]
    for s in sentences:
        toks=[w for w in tokenize(s) if w.isalpha() and w not in STOP]
        chunks.append(toks); counts.update(set(toks))
    rows=[]
    for i,(s,toks) in enumerate(zip(sentences,chunks)):
        recurrence=np.mean([math.log1p(counts[w]) for w in toks]) if toks else 0.0
        cross=np.mean([counts[w]>=3 for w in toks]) if toks else 0.0
        pred=-seq_loss(model,vocab.encode(s))
        rows.append({"idx":i,"recurrence":recurrence,"cross_context":cross,"predictability":pred})
    return pd.DataFrame(rows)

def evidence_scores(df):
    cols=["recurrence","cross_context","predictability"]
    X=df[cols].to_numpy(float)
    mu=X.mean(0);sd=X.std(0)
    active=sd>1e-8
    Z=np.zeros_like(X)
    Z[:,active]=(X[:,active]-mu[active])/(sd[active]+1e-9)

    C=np.zeros((3,3),float)
    idx=np.where(active)[0]
    if len(idx)>1:
        Ca=np.nan_to_num(np.abs(np.corrcoef(Z[:,idx],rowvar=False)))
        np.fill_diagonal(Ca,0.0)
        for ii,i in enumerate(idx):
            for jj,j in enumerate(idx):C[i,j]=Ca[ii,jj]

    rho=np.where(active,.70,0.0)
    a=rho**3
    if a.sum()<1e-12:return np.zeros(len(df)),np.zeros(3),C
    abar=a/(a.sum()+1e-12)
    red=C@abar
    w=a/(1+1.5*red);w=w/(w.sum()+1e-12)
    return Z@w,w,C

class ParityMemory:
    PRIME=1000003
    def __init__(self):self.items=[]
    @classmethod
    def chk(cls,ids):
        q=np.asarray(ids,dtype=np.int64)
        return int(q.sum()%cls.PRIME),int(((np.arange(len(q))+1)*q).sum()%cls.PRIME)
    def add(self,ids,idx):
        p,q=self.chk(ids)
        self.items.append({"primary":list(ids),"backup":list(ids),"p":p,"q":q,"idx":idx})
    def corrupt(self,prob,seed):
        rng=random.Random(seed)
        for it in self.items:
            if it["primary"] and rng.random()<prob:
                j=rng.randrange(len(it["primary"]))
                it["primary"][j]=(it["primary"][j]+rng.randint(1,23))%100000
    def retrieve(self,parity=True):
        out=[];repair=0;lost=0
        for it in self.items:
            if self.chk(it["primary"])==(it["p"],it["q"]):
                out.append(it["primary"]);continue
            if parity and self.chk(it["backup"])==(it["p"],it["q"]):
                out.append(it["backup"]);repair+=1
            else:lost+=1
        return out,repair,lost

def jaccard(a,b):
    if not a or not b:return 0.0
    return len(a&b)/(len(a|b)+1e-12)

def build_memory(stem,labels,vocab,model,keep_frac=.28):
    feat=chunk_features(stem,vocab,model)
    score,w,C=evidence_scores(feat)
    feat["score"]=score
    token_sets=[set(w for w in tokenize(s) if w.isalpha() and w not in STOP) for s in stem]

    # Evidence graph says duplicated/correlated reasons should not multiply.
    # Greedy diversity discount operationalizes that for near-duplicate chunks.
    k=max(1,int(round(keep_frac*len(stem))))
    selected=[];remain=set(range(len(stem)))
    lam=.85
    while remain and len(selected)<k:
        best=None;bestv=-1e99
        for i in remain:
            redundancy=max((jaccard(token_sets[i],token_sets[j]) for j in selected),default=0.0)
            v=float(score[i])-lam*redundancy
            if v>bestv:bestv=v;best=i
        selected.append(best);remain.remove(best)
    selected=set(selected)
    lo=np.quantile(score,.25)

    mem=ParityMemory();decisions=[]
    for i,s in enumerate(stem):
        if i in selected:
            decisions.append("INCLUDE");mem.add(vocab.encode(s),i)
        elif score[i]<=lo:decisions.append("EXCLUDE")
        else:decisions.append("VALUE_UNRESOLVED")
    feat["decision"]=decisions
    feat["eval_kind"]=[labels[i][0] for i in feat["idx"]]
    return mem,feat,{"weights":w.tolist(),"dependency":C.tolist(),"diversity_lambda":lam}

# ---------------------------------------------------------------------
# Experiment
# ---------------------------------------------------------------------
VARIANTS=["baseline","random_replay","framework","framework_no_parity"]

def run_condition(seed,condition_name,stream,stem,labels,device="cpu",fast=False):
    vocab=Vocab(stem+stream,max_vocab=8000)
    rows=[];memrows=[]

    for variant in VARIANTS:
        seed_all(seed)
        model=TinySSM(len(vocab.itos)).to(device)
        opt=torch.optim.Adam(model.parameters(),lr=.006)

        stem_epochs=2 if fast else 3
        order=list(range(len(stem)))
        for ep in range(stem_epochs):
            random.Random(seed+ep).shuffle(order)
            for i in order:train_sentence(model,opt,vocab.encode(stem[i]))

        pre=qa_metrics(model,vocab,False)
        pre_q=qa_metrics(model,vocab,True)

        mem,feat,ginfo=build_memory(stem,labels,vocab,model,.28)
        if variant=="random_replay":
            n=len(mem.items)
            picks=random.Random(seed+999).sample(range(len(stem)),n)
            mem=ParityMemory()
            for i in picks:mem.add(vocab.encode(stem[i]),i)

        src=[it["idx"] for it in mem.items]
        precision=float(np.mean([labels[i][0]=="fact" for i in src])) if src else 0.0
        distinct={(labels[i][2],labels[i][3]) for i in src if labels[i][0]=="fact"}
        coverage=len(distinct)/len(FACTS)

        mem.corrupt(.18,seed+12345)
        replay,repair,lost=mem.retrieve(parity=(variant!="framework_no_parity"))

        # One pass in fast proxy mode; two in full natural mode.
        epochs=1 if fast else 2
        checkpoints=[]
        total_steps=0
        for ep in range(epochs):
            # IMPORTANT: preserve condition order. We do not shuffle the
            # sequential-order conditions because order is the independent variable.
            for i,s in enumerate(stream):
                train_sentence(model,opt,vocab.encode(s))
                total_steps+=1
                if variant!="baseline" and replay and total_steps%6==0:
                    rid=replay[(total_steps//6)%len(replay)]
                    train_sentence(model,opt,rid)
                if i+1==len(stream)//2 and ep==0:
                    mid=qa_metrics(model,vocab,False)
                    midq=qa_metrics(model,vocab,True)
                    checkpoints.append((mid,midq))

        post=qa_metrics(model,vocab,False)
        post_q=qa_metrics(model,vocab,True)
        held=stream[-min(60,len(stream)):]
        dloss=float(np.mean([seq_loss(model,vocab.encode(s)) for s in held]))

        row={
            "seed":seed,"condition":condition_name,"variant":variant,
            "qa_pre_mrr":pre["mrr"],"qa_mid_mrr":checkpoints[0][0]["mrr"] if checkpoints else np.nan,
            "qa_post_mrr":post["mrr"],
            "heldoutq_pre_mrr":pre_q["mrr"],
            "heldoutq_mid_mrr":checkpoints[0][1]["mrr"] if checkpoints else np.nan,
            "heldoutq_post_mrr":post_q["mrr"],
            "qa_post_top1":post["top1"],"heldoutq_post_top1":post_q["top1"],
            "forgetting_mrr":pre["mrr"]-post["mrr"],
            "heldoutq_forgetting_mrr":pre_q["mrr"]-post_q["mrr"],
            "stream_loss":dloss,
            "memory_items":len(mem.items),
            "memory_fact_precision_eval_only":precision,
            "memory_fact_coverage_eval_only":coverage,
            "parity_repaired":repair,"parity_lost":lost,
        }
        rows.append(row)

        if variant=="framework":
            q=feat.copy()
            q["seed"]=seed;q["condition"]=condition_name
            memrows.append(q)

    return pd.DataFrame(rows),pd.concat(memrows,ignore_index=True)

def run_seed(seed=0,full=False,fast=False,device=None):
    seed_all(seed)
    stem,labels=make_stem_corpus(seed,repeats=5,junk=120)
    conditions,prov,A_name,B_name=corpus_conditions(seed,full=full,per_source=(90 if fast else 800))

    # hard provenance check: mixed branches must really contain both sources
    for name,(na,nb) in prov.items():
        if "only" not in name:
            assert na>0 and nb>0, (name,na,nb)

    if device is None:
        device="cuda" if torch.cuda.is_available() and full else "cpu"

    allr=[];allm=[]
    for name,stream in conditions.items():
        r,m=run_condition(seed,name,stream,stem,labels,device=device,fast=fast)
        allr.append(r);allm.append(m)

    result=pd.concat(allr,ignore_index=True)
    mem=pd.concat(allm,ignore_index=True)

    provenance_rows=[
        {"seed":seed,"condition":k,"source_A_name":A_name,"source_B_name":B_name,
         "source_A_count":v[0],"source_B_count":v[1],
         "stream_length":len(conditions[k]),
         "stream_sha256":hashlib.sha256("\n".join(conditions[k]).encode()).hexdigest()}
        for k,v in prov.items()
    ]
    return result,mem,pd.DataFrame(provenance_rows)

def aggregate(run_files):
    df=pd.concat([pd.read_csv(p) for p in run_files],ignore_index=True)
    summary=df.groupby(["condition","variant"])[
        ["qa_post_mrr","heldoutq_post_mrr","qa_post_top1","heldoutq_post_top1",
         "forgetting_mrr","heldoutq_forgetting_mrr","stream_loss",
         "memory_fact_precision_eval_only","memory_fact_coverage_eval_only",
         "parity_repaired","parity_lost"]
    ].agg(["mean","median","std","count"])
    return df,summary

def main(argv=None):
    ap=argparse.ArgumentParser()
    ap.add_argument("--full",action="store_true",
                    help="Use Tiny Shakespeare and public-domain KJV.")
    ap.add_argument("--fast",action="store_true",
                    help="Small proxy mechanics run.")
    ap.add_argument("--seed",type=int,default=0)
    ap.add_argument("--seeds",type=int,default=None)
    args=ap.parse_args(argv)

    if args.seeds is not None:
        seeds=list(range(args.seeds))
    else:
        seeds=[args.seed]

    run_paths=[];mem_paths=[];prov_paths=[]
    for seed in seeds:
        print("running seed",seed)
        r,m,p=run_seed(seed,full=args.full,fast=args.fast)
        tag="full" if args.full else "proxy"
        rp=OUT/f"{tag}_seed{seed}_runs.csv"
        mp=OUT/f"{tag}_seed{seed}_memory.csv"
        pp=OUT/f"{tag}_seed{seed}_provenance.csv"
        r.to_csv(rp,index=False);m.to_csv(mp,index=False);p.to_csv(pp,index=False)
        run_paths.append(rp);mem_paths.append(mp);prov_paths.append(pp)
        print("finished seed",seed)

    if len(run_paths)>1:
        df,summary=aggregate(run_paths)
        tag="full" if args.full else "proxy"
        df.to_csv(OUT/f"{tag}_{len(seeds)}seed_runs.csv",index=False)
        summary.to_csv(OUT/f"{tag}_{len(seeds)}seed_summary.csv")

        # Check order/interleaving and matched controls across every mixed condition.
        med=df.groupby(["condition","variant"]).median(numeric_only=True)
        mixed=[c for c in df.condition.unique() if "only" not in c]
        checks={}
        checks["all_mixed_conditions_exist"]=len(mixed)>=3
        checks["framework_beats_baseline_on_median_exact_QA_all_mixed"]=all(
            med.loc[(c,"framework"),"qa_post_mrr"] > med.loc[(c,"baseline"),"qa_post_mrr"] for c in mixed)
        checks["framework_beats_random_on_median_exact_QA_all_mixed"]=all(
            med.loc[(c,"framework"),"qa_post_mrr"] > med.loc[(c,"random_replay"),"qa_post_mrr"] for c in mixed)
        checks["parity_not_worse_on_median_exact_QA_all_mixed"]=all(
            med.loc[(c,"framework"),"qa_post_mrr"] >= med.loc[(c,"framework_no_parity"),"qa_post_mrr"] for c in mixed)
        checks["heldout_question_signal_survives_all_mixed"]=all(
            med.loc[(c,"framework"),"heldoutq_post_mrr"] > 0 for c in mixed)

        # Order sensitivity is scientifically allowed; we require that it is measured,
        # not that one order wins.
        order_conditions=[c for c in mixed if "_then_" in c]
        if len(order_conditions)>=2:
            vals=[med.loc[(c,"framework"),"qa_post_mrr"] for c in order_conditions]
            checks["order_effect_measured_nonidentical"]=bool(max(vals)-min(vals)>1e-6)

        (OUT/f"{tag}_{len(seeds)}seed_checks.json").write_text(json.dumps(checks,indent=2))

        report=f"""Adrianic v4.1 — Corrected Mixed-Corpus Order / Interleaving Transfer

Mode: {tag}
Seeds: {len(seeds)}

Why this version exists
=======================
v4.0's uploaded full report showed numerically identical Shakespeare and
"combined" branches. v4.1 fixes the construction so every mixed branch is built
from independently sampled source lists with explicit provenance counts and
hashes.

Conditions
==========
- source A only
- source B only
- A then B
- B then A
- balanced interleaving

The model still receives no semantic labels. The framework still receives no
fact/domain labels. Those labels remain evaluation-only.

New evaluation
==============
In addition to the original exact factual continuation probes, v4.1 includes a
small held-out rephrasing question set that is not used to construct the STEM
training stream. This is still a tiny language-model probe, not evidence of
human semantic understanding.

Summary
=======
{summary.to_string()}

Checks
======
{json.dumps(checks,indent=2)}

Scientific interpretation boundary
==================================
A positive result means selective deterministic memory survives genuine
two-corpus mixing and order changes better than matched controls. It does not
show that the model understands Shakespeare, the Bible, physics, or chemistry in
the human sense.

The key new question is whether previous factual structure remains usable while
the same learner is reorganized by different later corpus orders.
"""
        (OUT/f"{tag}_{len(seeds)}seed_REPORT.txt").write_text(report)
        print(report)



## Full natural-corpus run

Run this in Colab:

```python
main(["--full", "--seeds", "3"])
```

The code downloads Tiny Shakespeare and the public-domain KJV if they are not already present.

In [ ]:
main(["--full", "--seeds", "3"])

## Fast mechanics-only proxy run

```python
main(["--fast", "--seeds", "3"])
```

This uses two deliberately distinct synthetic text streams only to verify the order/interleaving machinery without network access.

## Verified local proxy result

```
Adrianic v4.1 — Corrected Mixed-Corpus Order / Interleaving Transfer
THREE-SEED LOCAL PROXY VERIFICATION

What this test verifies
=======================
This local run does NOT substitute synthetic streams for Shakespeare/Bible
semantics. It verifies the corrected mechanics before the natural-corpus Colab
run:

1. mixed streams really contain both independently constructed sources,
2. A->B and B->A are different sequential conditions,
3. balanced interleaving is genuinely balanced,
4. the deterministic replay framework is compared with baseline, random replay,
   and a no-parity control,
5. exact factual probes and held-out rephrasings are measured.

Mixed-condition median headline
===============================
           condition  baseline_exact_mrr  random_exact_mrr  framework_exact_mrr  framework_no_parity_exact_mrr  baseline_heldoutq_mrr  framework_heldoutq_mrr  framework_stream_loss  baseline_stream_loss
proxy_A_then_proxy_B            0.457301          0.714789             0.772529                       0.807606               0.129596                0.325022               0.598977              0.581928
proxy_B_then_proxy_A            0.553393          0.695521             0.759272                       0.780224               0.090288                0.218849               0.943354              0.906784
balanced_interleaved            0.560138          0.753958             0.819122                       0.825139               0.095853                0.283211               0.690503              0.702082

Provenance / construction check
===============================
 seed            condition source_A_name source_B_name  source_A_count  source_B_count  stream_length                                                    stream_sha256
    0         proxy_A_only       proxy_A       proxy_B              90               0             90 5539c8bdde92300324af73f30d493e38be747c001cd2ef26d8e026cd31700421
    0         proxy_B_only       proxy_A       proxy_B               0              90             90 fa08667062479a476f4abae4ebd83495134884c23b1e4f662aceb22be0d020c6
    0 proxy_A_then_proxy_B       proxy_A       proxy_B              90              90            180 14ba3071285242fca11a953ec4081ddcc240eff0589d991851c3fbfe0115b1e0
    0 proxy_B_then_proxy_A       proxy_A       proxy_B              90              90            180 098ec83343dd9b4620c18f505ea254cccf732bc141d8fafb93754cff7ecb6732
    0 balanced_interleaved       proxy_A       proxy_B              90              90            180 a141532e25ebc8c2d79850da13da5a7f21b2e633e8e66b34a1761a3d34682980
    1         proxy_A_only       proxy_A       proxy_B              90               0             90 c22679189e1f5962076c8e4c95193fe07e5145ef80aa81b851aebfe85528c9fb
    1         proxy_B_only       proxy_A       proxy_B               0              90             90 e7ceaee4891411244fd7e0fe7d12655ca6a8bdceea57f146ada551b0e70c95ed
    1 proxy_A_then_proxy_B       proxy_A       proxy_B              90              90            180 eeeba21d12f76a78a545a21483c64ac8528c3da26c1d74d7cd44e81f99c48edc
    1 proxy_B_then_proxy_A       proxy_A       proxy_B              90              90            180 32cac873beed713320f86b7c3a3f5dab93de9382a310b509030d4ddb73001151
    1 balanced_interleaved       proxy_A       proxy_B              90              90            180 84f7bee491f5a6105e555e697651906db39a127dd94a463a903733a5deae16b2
    2         proxy_A_only       proxy_A       proxy_B              90               0             90 55b44ab6bd88a7bbd2348d683efdb0fd0cb6efef80071ed11f8f1d3bc4c73f49
    2         proxy_B_only       proxy_A       proxy_B               0              90             90 afdd45cb33193137617eeaae45b35c21b866ccb4ef201baf7926b9c69a895600
    2 proxy_A_then_proxy_B       proxy_A       proxy_B              90              90            180 046e578d25dee5b7ed8a7288ca37c49e42508899fff3eacd94b5c216400216b0
    2 proxy_B_then_proxy_A       proxy_A       proxy_B              90              90            180 5962d7ba01d323787fde5ddffd6c42c977c8f8dac3d74a10d89e846bc5454bf4
    2 balanced_interleaved       proxy_A       proxy_B              90              90            180 38dfaadfe9cf1fa95abe29ea12185ac2de6333173247874514285c53976dabf1

Checks
======
{
  "mixed_provenance_has_both_sources": true,
  "mixed_stream_hashes_distinct_from_A_only": true,
  "framework_beats_baseline_exact_QA_all_mixed": true,
  "framework_beats_random_exact_QA_all_mixed": true,
  "framework_beats_baseline_heldout_Q_all_mixed": true,
  "parity_not_worse_exact_QA_all_mixed": false,
  "order_effect_nonidentical": true
}

Important result
================
The v4.0 combined-stream aliasing bug is fixed: each mixed stream has explicit
A/B source counts and a distinct content hash.

Across all three mixed proxy conditions, the deterministic framework retained
more exact STEM/factual continuation structure than both no replay and
capacity-matched random replay.

Held-out rephrasing performance also remained above baseline in all mixed
conditions, but this is still a very small word-level model and must not be
interpreted as human-like semantic understanding.

The A->B and B->A framework results are measurably different, so corpus order
does affect the retained state. v4.1 therefore exposes the variable we actually
wanted to study instead of accidentally duplicating a single corpus.

Natural-corpus next step
========================
Run the supplied Colab notebook with:

    main(["--full","--seeds","3"])

That performs the same corrected conditions using Tiny Shakespeare and the
public-domain KJV. The natural run, not this proxy mechanics run, is the test of
cross-domain language interference.

```